In [ ]:
from utils import load_data, fetch_top_docs, scrape_page, generate_search_query, evaluate_retrieval
from tqdm import tqdm
import json

### Generate dataset

In [ ]:
questions = load_data('NQ-open.train.jsonl')[0:500]
providers = [ 
            #  "ArXiv - arXiv.org",
             "Google News - news.google.com",
              "Europe PubMed Central - EuropePMC.org",
              "Internet Archive Items - archive.org",
             ] 

dataset = []
for i, q in tqdm(enumerate(questions)):
    docs = fetch_top_docs(q, providers, 5)
    dataset.append({"question": q, "documents": docs})
    
    if i% 10 == 0:
        print(f"Processed {i} questions")

        # Save dataset
        with open("qa_dataset.json", "w") as f:
            json.dump(dataset, f, indent=2)

# Save dataset
with open("qa_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
with open("qa_dataset.json", "r") as f:
    dataset = json.load(f)

for i, data in enumerate(dataset):
    
    content = None
    j = 0
    while content is None and j < len(data["documents"]):
        doc = data["documents"][j]
        content = scrape_page(doc)
        dataset[i]["documents"][j]["content"] = content
        j += 1
        
    if i% 10 == 0:
        print(f"Processed {i} documents")

        # Save dataset
        with open("qa_dataset_scrape.json", "w") as f:
            json.dump(dataset, f, indent=2)

In [11]:
#load dataset
with open("qa_dataset_scrape.json", "r") as f:
    dataset = json.load(f)

for data in dataset:
    if len(data["documents"]) == 0: # remove empty documents
        dataset.remove(data)     
    
    content = [doc["content"] for doc in data["documents"] if "content" in doc and doc["content"] is not None]
    if len(content) == 0: # remove empty documents
        dataset.remove(data)
           
    
with open("qa_dataset_scrape.json", "w") as f:
    json.dump(dataset, f, indent=2)

### Generate query

In [ ]:
import transformers
import torch

#load dataset
with open("qa_dataset_scrape.json", "r") as f:
    dataset = json.load(f)

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device_map="auto",
)

for i, data in enumerate(dataset):
    
    j=0
    text = None
    while text is None and j < len(data["documents"]):
        text = data["documents"][j]["content"]
        j+= 1
        
    messages = [
    {"role": "system", "content": "You are a helpful assistant that generates questions based on provided text. Output a single question, without any additional text and answer. Do not ask questions about cookies, privacy, or terms of service, only about the content of the text."},
    {"role": "user", "content": "Generate a simple question based on the following text: " + text + " ? Add a short context to the question if needed, for example, give a context about the film or book the question is about."},
    ]
    outputs = pipeline(
        messages,
        max_new_tokens=256,
    )
    print(outputs[0]["generated_text"][-1]["content"])

    dataset[i]["query"] = outputs[0]["generated_text"][-1]["content"]
    
    if i% 10 == 0:
        print(f"Processed {i} query")

        # Save dataset
        with open("qa_dataset_query.json", "w") as f:
            json.dump(dataset, f, indent=2)

with open("qa_dataset_query.json", "w") as f:
    json.dump(dataset, f, indent=2)

### Eval

In [ ]:
#load dataset
with open("qa_dataset_query.json", "r") as f:
    dataset = json.load(f)

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

dataset = generate_search_query(dataset, model_id, True)
        
        
with open("qa_dataset_llama3.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
with open("qa_dataset_query.json", "r") as f:
    dataset = json.load(f)

MODEL_PATH = "/capstor/store/cscs/swissai/infra01/swiss-alignment/checkpoints/Apertus3-8B_iter_1678000-tulu3-sft/checkpoint-13446"

dataset = generate_search_query(dataset, MODEL_PATH, False)
    
with open("qa_dataset_apertus.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
#load dataset
with open("qa_dataset_apertus8.json", "r") as f:
    dataset = json.load(f)
    
providers = [ 
            #  "ArXiv - arXiv.org",
             "Google News - news.google.com",
              "Europe PubMed Central - EuropePMC.org",
              "Internet Archive Items - archive.org",
             ] 

for i, data in enumerate(dataset):
    # if i < 70:
    #     continue
    docs = fetch_top_docs(dataset[i]["web_search_query"], providers=providers)
    dataset[i]["documents_answer"] = docs
    
    if i% 10 == 0:
        print(f"Processed {i} questions")

        # Save dataset
        with open("qa_dataset_res_apertus8.json", "w") as f:
            json.dump(dataset, f, indent=2)
    
with open("qa_dataset_res_apertus8.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
# Evaluate retrieval
retrieval_score = evaluate_retrieval(dataset)
print(f"Retrieval Score: {retrieval_score:.2f}")